# 04 Build datapoints (posts + their comments)

Joins the posts kept after the manual review (step 03) with all their comments.
Each post and each comment is one datapoint.

Comments are matched to posts through `link_id` (`t3_` + post id)

## Settings

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# input files and output folder (change these for each subreddit)
POSTS_FILE    = Path("path/to/Subreddit_manually_reviewed.csv")
COMMENTS_FILE = Path("path/to/Subreddit_comments_cleaned.csv")
OUT_DIR       = Path("path/to/output_folder")
CHUNK_SIZE    = 500_000   # comment rows read at once

## Load posts and their comments

In [ ]:
# load reviewed posts (sep=None so it works with , or ;)
posts = pd.read_csv(POSTS_FILE, sep=None, engine="python")
posts["join_key"] = "t3_" + posts["id"].astype(str)

# only keep comments from the reviewed posts (the comment file is read in chunks)
keys = set(posts["join_key"])
comments = pd.concat(
    c[c["link_id"].isin(keys)]
    for c in pd.read_csv(COMMENTS_FILE, dtype={"link_id": str, "parent_id": str, "id": str},
                         chunksize=CHUNK_SIZE))


def comment_level(parent_id, id_to_parent):
    # go up the parent chain until reaching the post
    level = 1
    while parent_id.startswith("t1_"):          # parent is a comment
        if parent_id not in id_to_parent or level > 200:
            return np.nan                       # chain broken (or a loop)
        parent_id = id_to_parent[parent_id]
        level += 1
    return level

# comment depth
id_to_parent = dict(zip("t1_" + comments["id"], comments["parent_id"]))
comments["comment_level"] = comments["parent_id"].map(lambda p: comment_level(p, id_to_parent))

## Build and save the datapoints
Saved as `datapoints.csv`.

In [ ]:
# post rows + comment rows (comments also get the post-only columns, e.g. debate labels)
post_rows = posts.assign(datapoint_type="post")
post_only = [c for c in posts.columns if c not in comments.columns]
comment_rows = (posts[post_only].merge(comments, left_on="join_key", right_on="link_id")
                .assign(datapoint_type="comment"))
comment_rows["datapoint"] = comment_rows["body_clean"]

# stack them, make ids, drop columns from earlier steps
dp = pd.concat([post_rows, comment_rows], ignore_index=True).drop(columns="join_key")
dp["datapoint_id"] = dp["datapoint_type"] + "_" + dp["id"].astype(str)
assert dp["datapoint_id"].is_unique, "datapoint_id collision"
dp = dp.drop(columns=[c for c in ["processed_text", "is_valid", "score_entry", "eligibility"] if c in dp.columns])

OUT_DIR.mkdir(parents=True, exist_ok=True)
dp.to_csv(OUT_DIR / "datapoints.csv", index=False)

## Summary
Saved as `datapoints_counts.csv` and `comment_levels.csv`

In [ ]:
# descriptive numbers
per_post = comments.groupby("link_id").size().reindex(posts["join_key"], fill_value=0)
authors = dp["author"].dropna()
per_author = authors[authors != "[deleted]"].value_counts()
post_auth = set(post_rows["author"]) - {"[deleted]"}
comm_auth = set(comment_rows["author"]) - {"[deleted]"}
counts = pd.Series({
    "total datapoints": len(dp),
    "post datapoints": len(post_rows),
    "comment datapoints": len(comment_rows),
    "posts with at least one comment": int((per_post > 0).sum()),
    "comments per post (mean)": round(per_post.mean(), 1),
    "comments per post (median)": per_post.median(),
    "comments per post (max)": per_post.max(),
    "unique authors (without [deleted])": len(post_auth | comm_auth),
    "post authors": len(post_auth),
    "comment authors": len(comm_auth),
    "authors in both": len(post_auth & comm_auth),
    "max datapoints by one author": int(per_author.max()) if len(per_author) else 0,
    "authors with > 5 datapoints": int((per_author > 5).sum()),
}, dtype=object)
display(counts)
counts.to_csv(OUT_DIR / "datapoints_counts.csv", header=["value"])

# comment depth (missing = parent chain broken by preprocessing)
levels = comments["comment_level"].value_counts(dropna=False).sort_index()
display(levels)
levels.to_csv(OUT_DIR / "comment_levels.csv", header=["comments"])